# Warren Meeting Slides (May 2026)

Generates the slide deck for the Warren White meeting per Ann's email feedback.

**Slide order (from Ann's email):**
1. Intro: dataset and problem
2. Data and tools used
3. Fabs (1/Mm) vs EC (ug/m3) for four SPARTAN sites
4. Fabs (1/Mm) vs FTIR EC for four sites
5. Fabs (1/Mm) vs Aethalometer
6. Iron (Fe and Fe/EC ranges, with rough IMPROVE comparison)
7. Seasonality + checklist of what we've ruled out
8. Four SPARTAN sites overlaid on IMPROVE background, EC mass on filter (ug)
9. Four separate plots, one per site, 5-95 percentile shading on both axes
10. White-style HIPS calibration space — split into 10a (raw R1/T1 + R-suppression vs tau), 10b (scaled t/r with t+r=1 locus), 10c (per-site collage), 10d (all four sites overlay). 10e is Warren's IMPROVE R/T figure if `WARREN_RT_PNG` is present.
11. Questions for Warren

IMPROVE FED data is loaded directly from the Google Drive folder used by the rest of the `improve_hips_offset/` notebooks. Override the location with the `AETHMODULAR_IMPROVE_DIR` environment variable if needed.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Repo paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPTS_DIR = REPO_ROOT / 'research' / 'ftir_hips_chem' / 'scripts'
for p in (SCRIPTS_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from config import SITES, MAC_VALUE  # noqa: E402
from data_matching import load_aethalometer_data, load_filter_data  # noqa: E402
from outliers import (  # noqa: E402
    apply_exclusion_flags,
    apply_threshold_flags,
    get_clean_data,
    print_exclusion_summary,
)

OUTPUT_DIR = REPO_ROOT / 'research' / 'ftir_hips_chem' / 'output' / 'warren_meeting'
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# IMPROVE FED workbooks live in Ahmad's Google Drive (same path the
# improve_hips_offset/ notebooks use). Override with AETHMODULAR_IMPROVE_DIR if needed.
DEFAULT_IMPROVE_DIR = Path(
    "/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/"
    "University/Research/Grad/UC Davis Ann/NASA MAIA/Data/Improve"
)
IMPROVE_DIR = Path(os.environ.get("AETHMODULAR_IMPROVE_DIR", DEFAULT_IMPROVE_DIR)).expanduser().resolve()

print(f'Repo: {REPO_ROOT}')
print(f'Output: {OUTPUT_DIR}')
print(f'IMPROVE input folder: {IMPROVE_DIR} (exists={IMPROVE_DIR.exists()})')

## Load SPARTAN data

For each of the four sites we build a per-filter dataframe with the columns we need on every slide:

- `FABS_invMm` - HIPS Fabs in inverse megameters (raw, **NOT** divided by MAC)
- `EC_ugm3` - FTIR EC concentration (ug/m^3)
- `EC_mass_ug` - EC mass on filter (ug). From `MassLoading_ug` in the unified filter dataset.
- `Fe_ugm3` - ChemSpec iron concentration
- `Aeth_ugm3` - Aethalometer IR BCc averaged over the filter day (ug/m^3)
- `Volume_m3`, `DepositArea_cm2`, `R`, `T`, `tau`, `t`, `r` - HIPS raw fields

In [ ]:
filter_data = load_filter_data()
aeth_data = load_aethalometer_data()

# FTIR/HIPS FilterId looks like 'ETAD-0001-1'; ChemSpec is 'ETAD-0001'.
# Pull the SITE-NNNN root with a positive regex so both forms collapse to it.
filter_data['base_filter_id'] = filter_data['FilterId'].astype(str).str.extract(
    r'^([A-Za-z]+-\d+)', expand=False
)

PARAMS_TO_PIVOT = [
    'EC_ftir', 'HIPS_Fabs', 'HIPS_T1', 'HIPS_R1',
    'HIPS_t', 'HIPS_r', 'HIPS_tau',
    'ChemSpec_Iron_PM2.5',
]


def build_site_table(site_name, site_code, df_aeth, filter_long):
    site_filters = filter_long[filter_long['Site'] == site_code].copy()
    if site_filters.empty:
        return None
    pool = site_filters[site_filters['Parameter'].isin(PARAMS_TO_PIVOT)].dropna(subset=['base_filter_id'])
    wide = (
        pool.pivot_table(index='base_filter_id',
                         columns='Parameter', values='Concentration', aggfunc='mean')
        .reset_index()
    )
    sample_date = (
        site_filters[['base_filter_id', 'SampleDate']]
        .dropna()
        .drop_duplicates('base_filter_id')
    )
    wide = wide.merge(sample_date, on='base_filter_id', how='left')
    meta = (
        site_filters[['base_filter_id', 'Volume_m3', 'DepositArea_cm2']]
        .groupby('base_filter_id')
        .agg(lambda s: s.dropna().iloc[0] if s.dropna().size else np.nan)
        .reset_index()
    )
    wide = wide.merge(meta, on='base_filter_id', how='left')
    ftype = (
        site_filters[['base_filter_id', 'FilterType']]
        .dropna()
        .drop_duplicates('base_filter_id')
    )
    wide = wide.merge(ftype, on='base_filter_id', how='left')
    ec_mass = (
        site_filters[site_filters['Parameter'].eq('EC_ftir')][['base_filter_id', 'MassLoading_ug']]
        .drop_duplicates('base_filter_id')
        .rename(columns={'MassLoading_ug': 'EC_mass_ug'})
    )
    wide = wide.merge(ec_mass, on='base_filter_id', how='left')
    wide = wide.rename(columns={
        'EC_ftir': 'EC_ugm3',
        'HIPS_Fabs': 'FABS_invMm',
        'ChemSpec_Iron_PM2.5': 'Fe_ngm3',  # ChemSpec metals are ng/m^3
        'HIPS_T1': 'T', 'HIPS_R1': 'R',
        'HIPS_t': 't', 'HIPS_r': 'r', 'HIPS_tau': 'tau',
    })
    if 'Fe_ngm3' in wide.columns:
        wide['Fe_ugm3'] = wide['Fe_ngm3'] / 1000.0  # ng/m3 -> ug/m3 for symmetry with EC
    wide['EC_mass_ug'] = wide['EC_mass_ug'].fillna(wide['EC_ugm3'] * wide['Volume_m3'])
    if df_aeth is not None and 'IR BCc' in df_aeth.columns:
        aeth_idx = df_aeth.set_index(pd.to_datetime(df_aeth['day_9am']))['IR BCc'].sort_index()
        def lookup(d):
            d = pd.to_datetime(d)
            sl = aeth_idx.loc[d - pd.Timedelta(days=1):d + pd.Timedelta(days=1)]
            return np.nan if sl.empty else sl.mean() / 1000.0
        wide['Aeth_ugm3'] = wide['SampleDate'].apply(lookup)
    else:
        wide['Aeth_ugm3'] = np.nan

    # Apply standard SPARTAN exclusion flow (per AGENTS.md):
    #   apply_exclusion_flags + apply_threshold_flags + get_clean_data.
    # The flag functions expect `date`, `aeth_bc`, `filter_ec` (all in ng/m3 to
    # match thresholds in outliers.MANUAL_OUTLIERS).
    wide['date'] = pd.to_datetime(wide['SampleDate'])
    wide['aeth_bc'] = wide['Aeth_ugm3'] * 1000.0
    wide['filter_ec'] = wide['EC_ugm3'] * 1000.0
    flagged = apply_exclusion_flags(wide, site_name)
    flagged = apply_threshold_flags(flagged, site_name)
    print_exclusion_summary(flagged, site_name)
    clean = get_clean_data(flagged).drop(columns=['aeth_bc', 'filter_ec'], errors='ignore')

    clean['site'] = site_name
    clean['site_color'] = SITES[site_name]['color']
    return clean


site_tables = {}
for site_name, cfg in SITES.items():
    df = build_site_table(site_name, cfg['code'], aeth_data.get(site_name), filter_data)
    if df is not None:
        site_tables[site_name] = df
        n_iron = df['Fe_ugm3'].notna().sum() if 'Fe_ugm3' in df.columns else 0
        n_iron_fabs = df.dropna(subset=['Fe_ugm3', 'FABS_invMm']).shape[0] if {'Fe_ugm3','FABS_invMm'}.issubset(df.columns) else 0
        n_blanks = (df['FilterType'] == 'FB').sum() if 'FilterType' in df.columns else 0
        print(f"{site_name}: clean rows={len(df)}, Fe={n_iron}, Fe&Fabs={n_iron_fabs}, blanks={n_blanks}")

## Load IMPROVE FED data

Reads the chemistry and RT workbooks from `IMPROVE_DIR`, joins them on the standard FED keys (`Dataset, SiteCode, POC, Date, AuxID`), filters to post-2003 per Warren's paper, and computes:

- `EC_ugm3` from `ECf_Val`, `FABS_invMm` from `fAbs_Val`, `Fe_ugm3` from `FEf_Val`
- `Volume_m3 = FlowRate_Val * SampDur_Val / 1000`
- `EC_mass_ug = EC_ugm3 * Volume_m3`
- `R = RefF_635_Val`, `T = TransF_635_Val`

Single-day extreme outliers (defined in `IMPROVE_DROP_SAMPLES`) are removed at load time so every slide that consumes IMPROVE data sees the same cleaned cloud. Currently dropping NOGA1 2024-01-01 (Fabs ≈ 311 1/Mm, ~10× the next highest IMPROVE point).

In [ ]:
IMPROVE_NA_VALUES = [-999, "-999", -999.0, "-999.0"]
IMPROVE_KEY_COLS = ["Dataset", "SiteCode", "POC", "Date", "AuxID"]

# IMPROVE rows to drop from the loader. These are extreme single-day outliers that
# stretch axes on the overlay slides. List entries are (SiteCode, 'YYYY-MM-DD').
IMPROVE_DROP_SAMPLES = [
    ("NOGA1", "2024-01-01"),  # Fabs ≈ 311 1/Mm, ~10x next highest IMPROVE point
]


def _find_improve_workbook(directory, required_cols):
    required_cols = set(required_cols)
    for path in sorted(Path(directory).glob("*.xlsx")):
        try:
            xl = pd.ExcelFile(path)
            if "Data" not in xl.sheet_names:
                continue
            cols = pd.read_excel(xl, sheet_name="Data", nrows=0).columns.astype(str).str.strip().tolist()
            if required_cols.issubset(cols):
                return path
        except Exception:
            continue
    raise FileNotFoundError(f"No IMPROVE workbook in {directory} contains {sorted(required_cols)}")


def _read_improve_data(path):
    df = pd.read_excel(path, sheet_name="Data", na_values=IMPROVE_NA_VALUES)
    df.columns = df.columns.astype(str).str.strip()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    for col in df.columns:
        if col.endswith("_Val") or col in {"POC", "AuxID"}:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def load_improve():
    chem_path = _find_improve_workbook(IMPROVE_DIR, ["SiteCode", "Date", "ECf_Val", "fAbs_Val"])
    rt_path = _find_improve_workbook(IMPROVE_DIR, ["SiteCode", "Date", "RefF_635_Val", "TransF_635_Val"])
    chem = _read_improve_data(chem_path)
    rt = _read_improve_data(rt_path)
    chem = chem.sort_values(IMPROVE_KEY_COLS).drop_duplicates(IMPROVE_KEY_COLS, keep="last")
    rt = rt.sort_values(IMPROVE_KEY_COLS).drop_duplicates(IMPROVE_KEY_COLS, keep="last")
    joined = chem.merge(rt, on=IMPROVE_KEY_COLS, how="left", suffixes=("", "_rt"), validate="one_to_one")

    out = pd.DataFrame({
        "Date": joined["Date"],
        "SiteCode": joined["SiteCode"],
        "EC_ugm3": joined["ECf_Val"],
        "FABS_invMm": joined["fAbs_Val"],
        "Fe_ugm3": joined.get("FEf_Val"),
        "R": joined.get("RefF_635_Val"),
        "T": joined.get("TransF_635_Val"),
    })
    flow = pd.to_numeric(joined.get("FlowRate_Val"), errors="coerce")
    dur = pd.to_numeric(joined.get("SampDur_Val"), errors="coerce")
    out["Volume_m3"] = np.where((flow > 0) & (dur > 0), flow * dur / 1000.0, np.nan)
    out["EC_mass_ug"] = out["EC_ugm3"] * out["Volume_m3"]
    out = out[out["Date"] >= pd.Timestamp("2003-01-01")]

    # Drop registered single-day outliers.
    n_before = len(out)
    for site_code, date_str in IMPROVE_DROP_SAMPLES:
        drop_mask = (out["SiteCode"] == site_code) & (out["Date"] == pd.Timestamp(date_str))
        if drop_mask.any():
            print(f"  Dropping IMPROVE outlier: {site_code} {date_str} "
                  f"(Fabs={out.loc[drop_mask, 'FABS_invMm'].iloc[0]:.1f} 1/Mm)")
            out = out[~drop_mask]
    if len(out) != n_before:
        print(f"  IMPROVE rows after outlier drop: {len(out):,} (was {n_before:,})")

    return out.reset_index(drop=True)


improve_df = load_improve()
print(f'IMPROVE rows loaded: {len(improve_df):,}')
print(f'  with EC & Fabs: {improve_df.dropna(subset=["EC_ugm3", "FABS_invMm"]).shape[0]:,}')
print(f'  with EC mass on filter: {improve_df["EC_mass_ug"].notna().sum():,}')
print(f'  with Fe: {improve_df["Fe_ugm3"].notna().sum():,}')
print(f'  with R/T: {improve_df.dropna(subset=["R", "T"]).shape[0]:,}')

In [ ]:
# Style helpers
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 15,
    'legend.fontsize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
})

SITE_ORDER = ['Addis_Ababa', 'Delhi', 'JPL', 'Beijing']
SITE_LABELS = {'Addis_Ababa': 'Addis Ababa', 'Delhi': 'Delhi', 'JPL': 'JPL/Pasadena', 'Beijing': 'Beijing'}
SITE_COLORS = {s: SITES[s]['color'] for s in SITE_ORDER}

def save_fig(fig, name):
    path = FIG_DIR / f'{name}.png'
    fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
    print(f'Saved: {path}')
    return path

## Slide 3 - Fabs (1/Mm) vs EC (ug/m3) for four sites

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
all_ec, all_fabs = [], []
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None or df.empty:
        continue
    sub = df.dropna(subset=['EC_ugm3', 'FABS_invMm'])
    sub = sub[(sub['EC_ugm3'] > 0) & (sub['FABS_invMm'] > 0)]
    if sub.empty: continue
    all_ec.append(sub['EC_ugm3'].values)
    all_fabs.append(sub['FABS_invMm'].values)
    ax.scatter(sub['EC_ugm3'], sub['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f'{SITE_LABELS[site]} (n={len(sub)})')
ax.set_xlabel('EC concentration (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('HIPS Fabs vs EC concentration - four SPARTAN sites')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
if all_ec:
    ax.set_xlim(0, np.nanpercentile(np.concatenate(all_ec), 99) * 1.1)
if all_fabs:
    ax.set_ylim(0, np.nanpercentile(np.concatenate(all_fabs), 99) * 1.1)
save_fig(fig, 'slide03_fabs_vs_ec_concentration')
plt.show()


## Slide 4 - Fabs (1/Mm) vs FTIR EC for four sites

Same comparison, FTIR EC explicitly. (If Slides 3 and 4 are intended to be the same, this is a duplicate kept per Ann's outline; otherwise edit this cell to swap in ChemSpec_EC for slide 3 above.)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
all_ec, all_fabs = [], []
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None or df.empty:
        continue
    sub = df.dropna(subset=['EC_ugm3', 'FABS_invMm'])
    sub = sub[(sub['EC_ugm3'] > 0) & (sub['FABS_invMm'] > 0)]
    if sub.empty: continue
    all_ec.append(sub['EC_ugm3'].values)
    all_fabs.append(sub['FABS_invMm'].values)
    ax.scatter(sub['EC_ugm3'], sub['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f'{SITE_LABELS[site]} (n={len(sub)})')
ax.set_xlabel('FTIR EC (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('HIPS Fabs vs FTIR EC - four SPARTAN sites')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
if all_ec:
    ax.set_xlim(0, np.nanpercentile(np.concatenate(all_ec), 99) * 1.1)
if all_fabs:
    ax.set_ylim(0, np.nanpercentile(np.concatenate(all_fabs), 99) * 1.1)
save_fig(fig, 'slide04_fabs_vs_ftir_ec')
plt.show()


## Slide 5 - Fabs (1/Mm) vs Aethalometer

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None or df.empty:
        continue
    sub = df.dropna(subset=['Aeth_ugm3', 'FABS_invMm'])
    if sub.empty:
        continue
    ax.scatter(sub['Aeth_ugm3'], sub['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=f"{SITE_LABELS[site]} (n={len(sub)})")
ax.set_xlabel('Aethalometer IR BCc (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('HIPS Fabs vs Aethalometer IR BCc')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
save_fig(fig, 'slide05_fabs_vs_aethalometer')
plt.show()


## Slide 6 - Iron (Fe and Fe/EC ranges)

Two panels:
- Left: Fe vs Fabs by site
- Right: Fe/EC ratio distributions by site

Add IMPROVE Fe and Fe/EC bands when IMPROVE data is loaded.

In [ ]:
# Drop Beijing Fe > 3 µg/m³ for this slide (likely contamination / dust event
# obscures the Fe vs Fabs comparison).
BEIJING_FE_MAX_UGM3 = 3.0


def _site_iron_view(df, site_name):
    if df is None:
        return df
    if site_name == 'Beijing' and 'Fe_ugm3' in df.columns:
        return df[~(df['Fe_ugm3'] > BEIJING_FE_MAX_UGM3)].copy()
    return df


fig, axes = plt.subplots(1, 2, figsize=(15, 7))

ax = axes[0]
for site in SITE_ORDER:
    df = _site_iron_view(site_tables.get(site), site)
    if df is None: continue
    sub = df.dropna(subset=['Fe_ugm3', 'FABS_invMm'])
    if sub.empty: continue
    label = f'{SITE_LABELS[site]} (n={len(sub)})'
    if site == 'Beijing':
        label += f', Fe ≤ {BEIJING_FE_MAX_UGM3:g} µg/m³'
    ax.scatter(sub['Fe_ugm3'], sub['FABS_invMm'], s=28, alpha=0.65,
               color=SITE_COLORS[site], label=label)
if 'Fe_ugm3' in improve_df.columns:
    fe_p5, fe_p95 = improve_df['Fe_ugm3'].quantile([0.05, 0.95])
    ax.axvspan(fe_p5, fe_p95, color='gray', alpha=0.18, label='IMPROVE Fe 5-95%')
ax.set_xlabel('Fe (µg/m³)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('Fe vs Fabs')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

ax = axes[1]
ratios, labels, colors = [], [], []
for site in SITE_ORDER:
    df = _site_iron_view(site_tables.get(site), site)
    if df is None: continue
    r = (df['Fe_ugm3'] / df['EC_ugm3']).replace([np.inf, -np.inf], np.nan).dropna()
    if r.empty: continue
    ratios.append(r.values); labels.append(SITE_LABELS[site]); colors.append(SITE_COLORS[site])
if ratios:
    bp = ax.boxplot(ratios, labels=labels, patch_artist=True, showfliers=False)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.6)
if {'Fe_ugm3', 'EC_ugm3'}.issubset(improve_df.columns):
    imp_ratio = (improve_df['Fe_ugm3'] / improve_df['EC_ugm3']).replace([np.inf, -np.inf], np.nan).dropna()
    p5, p95 = imp_ratio.quantile([0.05, 0.95])
    ax.axhspan(p5, p95, color='gray', alpha=0.2, label='IMPROVE Fe/EC 5-95%')
    ax.legend(fontsize=11)
ax.set_ylabel('Fe / EC')
ax.set_title('Fe/EC by site')
ax.grid(alpha=0.3, axis='y')
fig.suptitle(f'Iron - unlike pixelation, Fe should increase Fabs '
             f'(Beijing Fe capped at {BEIJING_FE_MAX_UGM3:g} µg/m³ for this view)',
             fontsize=14)
fig.tight_layout()
save_fig(fig, 'slide06_iron')
plt.show()

## Slide 7 - Seasonality + checklist

Boxplots of Fabs by season, per site. Below, a short checklist of variables already ruled out.

In [ ]:
from src.config.multi_site_seasons import SITE_SEASONS  # noqa: E402

fig, axes = plt.subplots(1, 4, figsize=(20, 6), sharey=True)
for ax, site in zip(axes, SITE_ORDER):
    df = site_tables.get(site)
    if df is None or 'SampleDate' not in df.columns:
        ax.set_visible(False); continue
    df = df.copy()
    df['month'] = pd.to_datetime(df['SampleDate']).dt.month
    seasons = SITE_SEASONS.get(site, {})
    data, labels, colors = [], [], []
    for sname, info in seasons.items():
        vals = df[df['month'].isin(info['months'])]['FABS_invMm'].dropna().values
        if len(vals) == 0: continue
        data.append(vals); labels.append(sname.split(' (')[0]); colors.append(info['color'])
    if data:
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
        for patch, c in zip(bp['boxes'], colors):
            patch.set_facecolor(c); patch.set_alpha(0.65)
    ax.set_title(SITE_LABELS[site])
    ax.grid(alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=20)
axes[0].set_ylabel('HIPS Fabs (1/Mm)')
fig.suptitle('Seasonality of HIPS Fabs by site', fontsize=15)
fig.tight_layout()
save_fig(fig, 'slide07_seasonality')
plt.show()

# Checklist of things tried/ruled out (rendered into the pptx as bullets)
RULED_OUT_CHECKLIST = [
    'Iron absorption (Fe / Fe-to-EC distribution)',
    'Seasonality - patterns are weaker for Addis than for Delhi/Beijing',
    'Smooth vs raw aethalometer (% difference)',
    'FTIR EC vs aethalometer agreement (no large offset)',
    'EC mass-loading range vs IMPROVE',
    'Filter sub-IDs / lot numbers as available',
    'HIPS MAC = 10 (not the source of the offset)',
    'Volume / flow ratio corrections (where applicable)',
]
print('\n'.join('- ' + x for x in RULED_OUT_CHECKLIST))


## Slide 8 - Four SPARTAN sites overlaid on IMPROVE background (EC mass on filter)

IMPROVE in gray (single-size dots, no shading); the four sites in distinct colors. This is the headline plot - we expect the SPARTAN scatter (especially Addis) to sit far above the IMPROVE cloud.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
sub = improve_df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=8, alpha=0.25,
           color='lightgray', label=f'IMPROVE (n={len(sub):,})', zorder=1)
for site in SITE_ORDER:
    df = site_tables.get(site)
    if df is None: continue
    sub = df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
    sub = sub[(sub['EC_mass_ug'] > 0) & (sub['FABS_invMm'] > 0)]
    if sub.empty: continue
    ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=36, alpha=0.8,
               color=SITE_COLORS[site], edgecolor='black', linewidth=0.4,
               label=f'{SITE_LABELS[site]} (n={len(sub)})', zorder=3)
ax.set_xlabel('EC mass on filter (µg)')
ax.set_ylabel('HIPS Fabs (1/Mm)')
ax.set_title('Four SPARTAN sites vs IMPROVE - EC mass on filter')
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(alpha=0.3)
# Cap x-axis at 99th-percentile across sites (drops single outlier filters)
all_mass = np.concatenate([d['EC_mass_ug'].dropna().values for d in site_tables.values()])
all_mass = all_mass[all_mass > 0]
if len(all_mass):
    ax.set_xlim(0, np.nanpercentile(all_mass, 99) * 1.1)
ax.set_ylim(bottom=0)
save_fig(fig, 'slide08_overlay_mass_on_filter')
plt.show()

## Slide 9 - Four separate plots, 5-95 percentile shading on both axes

One panel per SPARTAN site, IMPROVE in gray. Red/colored shading shows the 5-95 percentile of that site's EC-mass and Fabs simultaneously, to highlight where the bulk of the site's data lives versus the IMPROVE cloud.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
# Pre-compute shared axis caps from 99th percentile of pooled SPARTAN data.
all_mass = np.concatenate([d['EC_mass_ug'].dropna().values for d in site_tables.values()])
all_fabs = np.concatenate([d['FABS_invMm'].dropna().values for d in site_tables.values()])
x_max = np.nanpercentile(all_mass[all_mass > 0], 99) * 1.1 if len(all_mass) else 100
y_max = np.nanpercentile(all_fabs[all_fabs > 0], 99) * 1.1 if len(all_fabs) else 100

improve_xy = improve_df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])

for ax, site in zip(axes, SITE_ORDER):
    ax.scatter(improve_xy['EC_mass_ug'], improve_xy['FABS_invMm'], s=8, alpha=0.25,
               color='lightgray', label='IMPROVE', zorder=1)
    df = site_tables.get(site)
    if df is None:
        ax.set_xlim(0, x_max); ax.set_ylim(0, y_max); continue
    sub = df.dropna(subset=['EC_mass_ug', 'FABS_invMm'])
    sub = sub[(sub['EC_mass_ug'] > 0) & (sub['FABS_invMm'] > 0)]
    if not sub.empty:
        x_lo, x_hi = sub['EC_mass_ug'].quantile([0.05, 0.95])
        y_lo, y_hi = sub['FABS_invMm'].quantile([0.05, 0.95])
        ax.axvspan(x_lo, x_hi, color=SITE_COLORS[site], alpha=0.10, zorder=2)
        ax.axhspan(y_lo, y_hi, color=SITE_COLORS[site], alpha=0.10, zorder=2)
        ax.scatter(sub['EC_mass_ug'], sub['FABS_invMm'], s=36, alpha=0.85,
                   color=SITE_COLORS[site], edgecolor='black', linewidth=0.4,
                   label=f'{SITE_LABELS[site]} (n={len(sub)})', zorder=3)
    ax.set_xlabel('EC mass on filter (µg)')
    ax.set_ylabel('HIPS Fabs (1/Mm)')
    ax.set_title(SITE_LABELS[site])
    ax.legend(loc='upper left', fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, x_max)
    ax.set_ylim(0, y_max)
fig.suptitle('Per-site view with 5-95 percentile shading on both axes', fontsize=15)
fig.tight_layout()
save_fig(fig, 'slide09_per_site_shaded')
plt.show()

## Slide 10 - SPARTAN HIPS in White-style calibration space

Four figures (rendered as separate slides):

- **10a** - Headline 2-panel: raw `R1/T1` with the row/lot-specific calibration lines `T = a0 + a1·R`, and row-specific R-suppression vs stored `tau`.
- **10b** - Scaled HIPS coordinates (`r/t`) with the `t + r = 1` zero-absorption locus. Blanks sit on the line; loaded filters fall below.
- **10c** - Site-by-site 2×2 collage of the same raw `R1/T1` calibration space with per-site median annotations.
- **10d** - All four sites overlaid on a single axis so the Addis transmittance offset is obvious side-by-side.

If Warren's IMPROVE R/T figure (digitised from his 2024/2025 paper) is at `WARREN_RT_PNG`, the build cell adds it as **10e**.

In [ ]:
WARREN_RT_PNG = REPO_ROOT / 'tmp_warren_pages' / 'warren_RT.png'

# ── Load raw HIPS export (T1, R1, t, r, tau, Intercept, Slope, LotId) ────────
DEFAULT_DRIVE_ROOT = (
    Path.home() / 'Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive'
    / 'University/Research/Grad/UC Davis Ann/NASA MAIA/Data'
)
DRIVE_ROOT = Path(os.environ.get('AETHMODULAR_MAIA_DATA_ROOT', DEFAULT_DRIVE_ROOT)).expanduser().resolve()
RAW_HIPS_PATH = DRIVE_ROOT / 'Combine csv files' / 'Raw Datafiles' / 'Four_Sites_HIPS_data.csv'
SITE_CODE_TO_NAME = {'CHTS': 'Beijing', 'INDH': 'Delhi', 'USPA': 'JPL', 'ETAD': 'Addis_Ababa'}

hips_raw = pd.read_csv(RAW_HIPS_PATH)
hips_raw['SampleDate'] = pd.to_datetime(hips_raw['SampleDate'], errors='coerce')
hips_raw['SiteName'] = hips_raw['Site'].map(SITE_CODE_TO_NAME)
hips_raw['CoefPair'] = hips_raw['Intercept'].map('{:.1f}'.format) + ' / ' + hips_raw['Slope'].map('{:.3f}'.format)
hips_raw['t_plus_r'] = hips_raw['t'] + hips_raw['r']

# Apply standard SPARTAN exclusions to PM2.5 samples (field blanks are kept).
hips_raw['date'] = hips_raw['SampleDate']
keep_parts = []
for site_name in SITE_ORDER:
    site_part = hips_raw[hips_raw['SiteName'] == site_name].copy()
    if site_part.empty:
        continue
    flagged = apply_exclusion_flags(site_part, site_name)
    excl_mask = flagged['is_excluded'] & (flagged['FilterType'] == 'PM2.5')
    keep_parts.append(flagged.loc[~excl_mask])
# Keep any rows from sites not in SITE_ORDER (none expected) untouched.
extra = hips_raw[~hips_raw['SiteName'].isin(SITE_ORDER)]
hips_raw = pd.concat(keep_parts + [extra], ignore_index=True)

hips_geom = hips_raw.dropna(subset=['T1', 'R1', 'tau', 'Intercept', 'Slope', 't', 'r']).copy()
hips_geom['R_expected'] = (hips_geom['T1'] - hips_geom['Intercept']) / hips_geom['Slope']
hips_geom['R_suppression_pct'] = (1 - hips_geom['R1'] / hips_geom['R_expected']) * 100

calibration_lines = (
    hips_geom.groupby(['LotId', 'Intercept', 'Slope', 'CoefPair'])
    .agg(
        n_filters=('FilterId', 'count'),
        n_blanks=('FilterType', lambda s: (s == 'FB').sum()),
        n_samples=('FilterType', lambda s: (s == 'PM2.5').sum()),
    )
    .reset_index()
    .sort_values('n_filters', ascending=False)
)
line_styles = ['-', '--', '-.', ':']
line_colors = ['black', '#555555', '#117A65', '#7B241C', '#884EA0']
line_style_map = {
    row.CoefPair: (line_colors[i % len(line_colors)], line_styles[i % len(line_styles)])
    for i, row in calibration_lines.reset_index(drop=True).iterrows()
}
r_line = np.linspace(40, 520, 300)

# Map our SITE_LABELS / SITE_COLORS keys onto the SiteName column used here.
def _site_filter(df, site_key, ftype=None):
    sub = df[df['SiteName'] == site_key]
    return sub if ftype is None else sub[sub['FilterType'] == ftype]

addis_samples = _site_filter(hips_geom, 'Addis_Ababa', 'PM2.5')
addis_med = addis_samples[['T1', 'R1', 'R_expected', 'R_suppression_pct', 'tau']].median()

In [ ]:
# ── Slide 10a: 2-panel White-style headline figure ──────────────────────────
# (Scaled t/r panel is broken out into slide 10b.)
fig, axes = plt.subplots(1, 2, figsize=(16, 6.8))

# Panel A: raw R1 vs T1 with calibration lines
ax = axes[0]
for site in SITE_ORDER:
    color = SITE_COLORS[site]
    blanks = _site_filter(hips_geom, site, 'FB')
    samples = _site_filter(hips_geom, site, 'PM2.5')
    ax.scatter(samples['R1'], samples['T1'], s=22, color=color, alpha=0.42,
               edgecolor='none', label=f'{SITE_LABELS[site]} PM2.5')
    ax.scatter(blanks['R1'], blanks['T1'], s=46, color=color, marker='s', alpha=0.82,
               edgecolor='black', linewidth=0.35)
for _, row in calibration_lines.iterrows():
    color, ls = line_style_map[row['CoefPair']]
    t_line = row['Intercept'] + row['Slope'] * r_line
    visible = (t_line >= -50) & (t_line <= 1150)
    label = f"a0={row['Intercept']:.1f}, a1={row['Slope']:.3f} (n={int(row['n_filters'])})"
    lw = 2.0 if row['n_filters'] == calibration_lines['n_filters'].max() else 1.35
    ax.plot(r_line[visible], t_line[visible], color=color, ls=ls, lw=lw, alpha=0.9, label=label)
ax.scatter(addis_med['R1'], addis_med['T1'], marker='*', s=210, color='crimson',
           edgecolor='white', linewidth=0.8, zorder=7)
ax.annotate('', xy=(addis_med['R1'], addis_med['T1']),
            xytext=(addis_med['R_expected'], addis_med['T1']),
            arrowprops=dict(arrowstyle='<->', color='crimson', lw=2.0))
ax.annotate(
    f"Addis median\nR actual={addis_med['R1']:.0f}; blank-model R={addis_med['R_expected']:.0f}\n"
    f"R suppression={addis_med['R_suppression_pct']:.0f}%\ntau={addis_med['tau']:.2f}",
    xy=((addis_med['R1'] + addis_med['R_expected']) / 2, addis_med['T1']),
    xytext=(30, -95), textcoords='offset points', fontsize=8.5, color='crimson',
    arrowprops=dict(arrowstyle='->', color='crimson', lw=1.0),
    bbox=dict(boxstyle='round,pad=0.30', facecolor='white', edgecolor='crimson', alpha=0.88),
)
ax.set_xlabel('Raw HIPS sphere signal R1')
ax.set_ylabel('Raw HIPS plate signal T1')
ax.set_title('White-style raw HIPS calibration space\nT = a0 + a1·R')
ax.set_xlim(40, 520); ax.set_ylim(0, 1100)
ax.grid(alpha=0.22)
ax.legend(fontsize=7.5, loc='upper right', frameon=True, ncol=1)

# Panel B: row-specific R suppression vs tau
ax = axes[1]
for site in SITE_ORDER:
    color = SITE_COLORS[site]
    s = _site_filter(hips_geom, site, 'PM2.5')
    ax.scatter(s['R_suppression_pct'], s['tau'], s=24, color=color, alpha=0.55,
               edgecolor='none', label=SITE_LABELS[site])
s_vals = np.linspace(0, 0.95, 240)
for T_ref, R_ref, lbl, alpha in [
    (900, 185, 'Theory: light loading', 0.55),
    (600, 295, 'Theory: mid loading', 0.8),
    (390, 370, 'Theory: Addis-like T', 0.8),
]:
    tau_theory = np.log(1 + 2.783 * R_ref * s_vals / T_ref)
    ax.plot(s_vals * 100, tau_theory, lw=1.5, ls='--', color='black', alpha=alpha, label=lbl)
ax.axvline(addis_med['R_suppression_pct'], color='crimson', ls=':', lw=1.5)
ax.axhline(addis_med['tau'], color='crimson', ls=':', lw=1.5)
ax.set_xlabel('R suppression vs row-specific blank model (%)')
ax.set_ylabel('Stored HIPS tau')
ax.set_title('R suppression tracks tau when row-specific a0/a1 are used')
ax.set_xlim(-10, 100); ax.set_ylim(-0.05, 2.3)
ax.grid(alpha=0.22)
ax.legend(fontsize=7.5, loc='upper left', frameon=True)

fig.suptitle('SPARTAN HIPS in the same calibration space used by White et al. (2025)',
             fontsize=15, y=1.02)
fig.tight_layout()
save_fig(fig, 'slide10a_white_calibration_space')
plt.show()

In [ ]:
# ── Slide 10b: scaled HIPS coordinates with t+r=1 zero-absorption locus ─────
fig, ax = plt.subplots(figsize=(9.5, 8))
for site in SITE_ORDER:
    color = SITE_COLORS[site]
    blanks = _site_filter(hips_geom, site, 'FB')
    samples = _site_filter(hips_geom, site, 'PM2.5')
    ax.scatter(samples['r'], samples['t'], s=32, color=color, alpha=0.55,
               edgecolor='none', label=f'{SITE_LABELS[site]} PM2.5 (n={len(samples)})')
    if not blanks.empty:
        ax.scatter(blanks['r'], blanks['t'], s=58, color=color, marker='s', alpha=0.9,
                   edgecolor='black', linewidth=0.4,
                   label=f'{SITE_LABELS[site]} field blank (n={len(blanks)})')
r_scaled = np.linspace(0, 0.9, 200)
ax.plot(r_scaled, 1 - r_scaled, color='black', lw=2.6, label='Zero-absorption: t + r = 1')
ax.axhspan(0, 0.05, color='crimson', alpha=0.07, label='near T floor')
ax.set_xlabel('Scaled sphere signal r')
ax.set_ylabel('Scaled plate signal t')
ax.set_title('Scaled HIPS coordinates — blanks sit on t + r = 1; loaded filters fall below')
ax.set_xlim(0.20, 0.85); ax.set_ylim(0.0, 0.85)
ax.grid(alpha=0.25)
ax.legend(fontsize=10, loc='upper right', frameon=True)
fig.tight_layout()
save_fig(fig, 'slide10b_scaled_hips')
plt.show()

In [ ]:
# ── Slide 10c: site-by-site collage of raw R1/T1 calibration space ───────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharex=True, sharey=True)
axes = axes.ravel()
for ax, site in zip(axes, SITE_ORDER):
    color = SITE_COLORS[site]
    site_df = hips_geom[hips_geom['SiteName'] == site]
    blanks = site_df[site_df['FilterType'] == 'FB']
    samples = site_df[site_df['FilterType'] == 'PM2.5']
    ax.scatter(samples['R1'], samples['T1'], s=26, color=color, alpha=0.48,
               edgecolor='none', label=f'PM2.5 (n={len(samples)})')
    ax.scatter(blanks['R1'], blanks['T1'], s=58, color=color, marker='s', alpha=0.88,
               edgecolor='black', linewidth=0.35, label=f'Field blank (n={len(blanks)})')
    for _, row in calibration_lines.iterrows():
        if row['CoefPair'] not in site_df['CoefPair'].unique():
            continue
        line_color, ls = line_style_map[row['CoefPair']]
        t_line = row['Intercept'] + row['Slope'] * r_line
        visible = (t_line >= -50) & (t_line <= 1150)
        ax.plot(r_line[visible], t_line[visible], color=line_color, ls=ls, lw=1.45, alpha=0.78)
    if not samples.empty:
        med = samples[['T1', 'R1', 'R_expected', 'R_suppression_pct', 'tau']].median()
        ax.scatter(med['R1'], med['T1'], marker='*', s=155, color='crimson',
                   edgecolor='white', linewidth=0.7, zorder=6)
        ax.annotate('', xy=(med['R1'], med['T1']),
                    xytext=(med['R_expected'], med['T1']),
                    arrowprops=dict(arrowstyle='<->', color='crimson', lw=1.5))
        ax.text(
            0.03, 0.04,
            f"median R suppression={med['R_suppression_pct']:.0f}%\nmedian tau={med['tau']:.2f}",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.30', facecolor='white', edgecolor='0.85', alpha=0.88),
        )
    ax.set_title(SITE_LABELS[site])
    ax.grid(alpha=0.22)
    ax.legend(loc='upper right', fontsize=9, frameon=True)

for ax in axes[2:]:
    ax.set_xlabel('Raw HIPS sphere signal R1')
for ax in axes[::2]:
    ax.set_ylabel('Raw HIPS plate signal T1')
for ax in axes:
    ax.set_xlim(40, 520); ax.set_ylim(0, 1100)
fig.suptitle('Per site: where each filter sits relative to its row/lot calibration line',
             fontsize=15, y=1.01)
fig.tight_layout()
save_fig(fig, 'slide10c_site_panels')
plt.show()

In [ ]:
# ── Slide 10d: all four sites overlaid on a single axis ─────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
for site in SITE_ORDER:
    color = SITE_COLORS[site]
    samples = _site_filter(hips_geom, site, 'PM2.5')
    blanks = _site_filter(hips_geom, site, 'FB')
    ax.scatter(samples['R1'], samples['T1'], s=28, color=color, alpha=0.6,
               edgecolor='none', label=f'{SITE_LABELS[site]} (n={len(samples)})')
    if not blanks.empty:
        ax.scatter(blanks['R1'], blanks['T1'], s=55, color=color, marker='s', alpha=0.9,
                   edgecolor='black', linewidth=0.35)
# Overlay calibration lines (lightly) for context.
for _, row in calibration_lines.iterrows():
    line_color, ls = line_style_map[row['CoefPair']]
    t_line = row['Intercept'] + row['Slope'] * r_line
    visible = (t_line >= -50) & (t_line <= 1150)
    ax.plot(r_line[visible], t_line[visible], color=line_color, ls=ls, lw=1.0, alpha=0.45)
ax.set_xlim(40, 520); ax.set_ylim(0, 1100)
ax.set_xlabel('Raw HIPS sphere signal R1')
ax.set_ylabel('Raw HIPS plate signal T1')
ax.set_title('All four sites on shared axes — Addis sits at lower T than the others')
ax.legend(fontsize=11, ncol=2, loc='upper right', frameon=True)
ax.grid(alpha=0.22)
fig.tight_layout()
save_fig(fig, 'slide10d_all_sites_overlay')
plt.show()

if WARREN_RT_PNG.exists():
    print(f"Warren's IMPROVE R/T figure available at: {WARREN_RT_PNG}")
else:
    print(f"Warren's IMPROVE R/T figure NOT FOUND at {WARREN_RT_PNG}.\n"
          "Drop a PNG there to include it as the comparison panel.")

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)
BLANK = prs.slide_layouts[6]

def add_title(slide, text, size=28):
    box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.5), Inches(0.7))
    tf = box.text_frame; tf.word_wrap = True
    tf.text = text
    for p in tf.paragraphs:
        for r in p.runs: r.font.size = Pt(size); r.font.bold = True

def add_image(slide, png_path, left=0.4, top=1.0, width=12.5, height=6.2):
    if Path(png_path).exists():
        slide.shapes.add_picture(str(png_path), Inches(left), Inches(top),
                                 width=Inches(width), height=Inches(height))
    else:
        box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
        box.text_frame.text = f'[FIGURE MISSING: {png_path}]'

def add_bullets(slide, items, left=0.5, top=1.2, width=12.3, height=5.8, size=20):
    box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = box.text_frame; tf.word_wrap = True
    for i, item in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = '• ' + item
        for r in p.runs: r.font.size = Pt(size)

# Slide 1 - intro
s = prs.slides.add_slide(BLANK)
add_title(s, 'HIPS Fabs offset at Addis Ababa - meeting with Warren White', size=30)
add_bullets(s, [
    'Goal: figure out why HIPS Fabs in Addis is anomalously high vs FTIR EC and aethalometer',
    'Four SPARTAN sites: Addis Ababa, Delhi, JPL/Pasadena, Beijing',
    'Comparison reference: post-2003 IMPROVE network (Warren et al. paper)',
    'Key question: is the offset consistent with the pixelation effect Warren describes,',
    '   or is something else going on?',
])

# Slide 2 - data and tools
s = prs.slides.add_slide(BLANK)
add_title(s, 'Data and tools')
add_bullets(s, [
    'SPARTAN filters: HIPS (Fabs, R, T, tau, t, r), FTIR EC, ChemSpec metals, ions',
    'Aethalometer (AE33): IR BCc, smoothed and raw',
    'IMPROVE (FED Wizard extract, post-2003): EC, Fabs, Fe, Volume; no R/T, no field blanks, no filter IDs',
    'Matching: filter date ± 1 day to aethalometer day_9am window',
    'MAC = 10 m²/g for HIPS → BC conversion (raw 1/Mm shown here)',
])

# Slides 3-7 - each is one figure
for slide_num, (png, title) in enumerate([
    ('slide03_fabs_vs_ec_concentration', 'Fabs (1/Mm) vs EC (µg/m³) - four sites'),
    ('slide04_fabs_vs_ftir_ec', 'Fabs (1/Mm) vs FTIR EC - four sites'),
    ('slide05_fabs_vs_aethalometer', 'Fabs (1/Mm) vs Aethalometer IR BCc'),
    ('slide06_iron', 'Iron - Fe and Fe/EC ranges'),
    ('slide07_seasonality', 'Seasonality and what we have ruled out'),
], start=3):
    s = prs.slides.add_slide(BLANK)
    add_title(s, f'{slide_num}. {title}')
    add_image(s, FIG_DIR / f'{png}.png')

# Slide 7 - append checklist as a small text box at the bottom
checklist_slide = prs.slides[6]  # the seasonality slide
box = checklist_slide.shapes.add_textbox(Inches(0.4), Inches(6.4), Inches(12.5), Inches(1.0))
tf = box.text_frame; tf.word_wrap = True
tf.text = 'Ruled out / inconclusive: ' + '; '.join(RULED_OUT_CHECKLIST)
for p in tf.paragraphs:
    for r in p.runs: r.font.size = Pt(11); r.font.italic = True

# Slide 8 - overlay
s = prs.slides.add_slide(BLANK)
add_title(s, '8. Four SPARTAN sites overlaid on IMPROVE - EC mass on filter')
add_image(s, FIG_DIR / 'slide08_overlay_mass_on_filter.png')

# Slide 9 - per-site shaded
s = prs.slides.add_slide(BLANK)
add_title(s, '9. Per site, 5-95 percentile shading on both axes')
add_image(s, FIG_DIR / 'slide09_per_site_shaded.png')

# Slide 10a - White-style 2-panel headline
s = prs.slides.add_slide(BLANK)
add_title(s, '10a. SPARTAN HIPS in White-style calibration space')
add_image(s, FIG_DIR / 'slide10a_white_calibration_space.png')

# Slide 10b - scaled HIPS (t+r=1 zero-absorption locus)
s = prs.slides.add_slide(BLANK)
add_title(s, '10b. Scaled HIPS coordinates — t + r = 1 zero-absorption locus')
add_image(s, FIG_DIR / 'slide10b_scaled_hips.png')

# Slide 10c - per-site collage
s = prs.slides.add_slide(BLANK)
add_title(s, '10c. Per-site collage — each filter vs its calibration line')
add_image(s, FIG_DIR / 'slide10c_site_panels.png')

# Slide 10d - all sites overlay
s = prs.slides.add_slide(BLANK)
add_title(s, '10d. All four sites overlaid — Addis transmittance offset')
add_image(s, FIG_DIR / 'slide10d_all_sites_overlay.png')

if WARREN_RT_PNG.exists():
    # add separate slide for Warren's IMPROVE R/T comparison
    s = prs.slides.add_slide(BLANK)
    add_title(s, '10e. Warren et al. - IMPROVE R/T (for comparison)')
    add_image(s, WARREN_RT_PNG)

# Slide 11 - questions for Warren
s = prs.slides.add_slide(BLANK)
add_title(s, '11. Questions for Warren')
add_bullets(s, [
    'Does the Addis Fabs offset look like an extreme version of the pixelation effect, or something different?',
    'Pixelation predicts UNDER-estimation at high loadings - we see OVER-estimation. Reconcile?',
    'For SPARTAN R/T values that fall outside the IMPROVE envelope, do those filters look physically plausible to you?',
    'Do you have residual / unpublished tests on chemical-composition impacts (Fe, dust) that would be worth re-running on Addis?',
    'Is there a sensible IMPROVE subset (high-load wildfire days?) you would compare Addis against?',
])

out_pptx = OUTPUT_DIR / 'warren_meeting_slides.pptx'
prs.save(out_pptx)
print(f'Saved deck: {out_pptx}')

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)
BLANK = prs.slide_layouts[6]

def add_title(slide, text, size=28):
    box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.5), Inches(0.7))
    tf = box.text_frame; tf.word_wrap = True
    tf.text = text
    for p in tf.paragraphs:
        for r in p.runs: r.font.size = Pt(size); r.font.bold = True

def add_image(slide, png_path, left=0.4, top=1.0, width=12.5, height=6.2):
    if Path(png_path).exists():
        slide.shapes.add_picture(str(png_path), Inches(left), Inches(top),
                                 width=Inches(width), height=Inches(height))
    else:
        box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
        box.text_frame.text = f'[FIGURE MISSING: {png_path}]'

def add_bullets(slide, items, left=0.5, top=1.2, width=12.3, height=5.8, size=20):
    box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = box.text_frame; tf.word_wrap = True
    for i, item in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = '• ' + item
        for r in p.runs: r.font.size = Pt(size)

# Slide 1 - intro
s = prs.slides.add_slide(BLANK)
add_title(s, 'HIPS Fabs offset at Addis Ababa - meeting with Warren White', size=30)
add_bullets(s, [
    'Goal: figure out why HIPS Fabs in Addis is anomalously high vs FTIR EC and aethalometer',
    'Four SPARTAN sites: Addis Ababa, Delhi, JPL/Pasadena, Beijing',
    'Comparison reference: post-2003 IMPROVE network (Warren et al. paper)',
    'Key question: is the offset consistent with the pixelation effect Warren describes,',
    '   or is something else going on?',
])

# Slide 2 - data and tools
s = prs.slides.add_slide(BLANK)
add_title(s, 'Data and tools')
add_bullets(s, [
    'SPARTAN filters: HIPS (Fabs, R, T, tau, t, r), FTIR EC, ChemSpec metals, ions',
    'Aethalometer (AE33): IR BCc, smoothed and raw',
    'IMPROVE (FED Wizard extract, post-2003): EC, Fabs, Fe, Volume; no R/T, no field blanks, no filter IDs',
    'Matching: filter date ± 1 day to aethalometer day_9am window',
    'MAC = 10 m²/g for HIPS → BC conversion (raw 1/Mm shown here)',
])

# Slides 3-7 - each is one figure
for slide_num, (png, title) in enumerate([
    ('slide03_fabs_vs_ec_concentration', 'Fabs (1/Mm) vs EC (µg/m³) - four sites'),
    ('slide04_fabs_vs_ftir_ec', 'Fabs (1/Mm) vs FTIR EC - four sites'),
    ('slide05_fabs_vs_aethalometer', 'Fabs (1/Mm) vs Aethalometer IR BCc'),
    ('slide06_iron', 'Iron - Fe and Fe/EC ranges'),
    ('slide07_seasonality', 'Seasonality and what we have ruled out'),
], start=3):
    s = prs.slides.add_slide(BLANK)
    add_title(s, f'{slide_num}. {title}')
    add_image(s, FIG_DIR / f'{png}.png')

# Slide 7 - append checklist as a small text box at the bottom
checklist_slide = prs.slides[6]  # the seasonality slide
box = checklist_slide.shapes.add_textbox(Inches(0.4), Inches(6.4), Inches(12.5), Inches(1.0))
tf = box.text_frame; tf.word_wrap = True
tf.text = 'Ruled out / inconclusive: ' + '; '.join(RULED_OUT_CHECKLIST)
for p in tf.paragraphs:
    for r in p.runs: r.font.size = Pt(11); r.font.italic = True

# Slide 8 - overlay
s = prs.slides.add_slide(BLANK)
add_title(s, '8. Four SPARTAN sites overlaid on IMPROVE - EC mass on filter')
add_image(s, FIG_DIR / 'slide08_overlay_mass_on_filter.png')

# Slide 9 - per-site shaded
s = prs.slides.add_slide(BLANK)
add_title(s, '9. Per site, 5-95 percentile shading on both axes')
add_image(s, FIG_DIR / 'slide09_per_site_shaded.png')

# Slide 10a - White-style 3-panel headline
s = prs.slides.add_slide(BLANK)
add_title(s, '10a. SPARTAN HIPS in White-style calibration space')
add_image(s, FIG_DIR / 'slide10a_white_calibration_space.png')

# Slide 10b - per-site collage
s = prs.slides.add_slide(BLANK)
add_title(s, '10b. Per-site collage — each filter vs its calibration line')
add_image(s, FIG_DIR / 'slide10b_site_panels.png')

# Slide 10c - all sites overlay
s = prs.slides.add_slide(BLANK)
add_title(s, '10c. All four sites overlaid — Addis transmittance offset')
add_image(s, FIG_DIR / 'slide10c_all_sites_overlay.png')

if WARREN_RT_PNG.exists():
    # add separate slide for Warren's IMPROVE R/T comparison
    s = prs.slides.add_slide(BLANK)
    add_title(s, '10d. Warren et al. - IMPROVE R/T (for comparison)')
    add_image(s, WARREN_RT_PNG)

# Slide 11 - questions for Warren
s = prs.slides.add_slide(BLANK)
add_title(s, '11. Questions for Warren')
add_bullets(s, [
    'Does the Addis Fabs offset look like an extreme version of the pixelation effect, or something different?',
    'Pixelation predicts UNDER-estimation at high loadings - we see OVER-estimation. Reconcile?',
    'For SPARTAN R/T values that fall outside the IMPROVE envelope, do those filters look physically plausible to you?',
    'Do you have residual / unpublished tests on chemical-composition impacts (Fe, dust) that would be worth re-running on Addis?',
    'Is there a sensible IMPROVE subset (high-load wildfire days?) you would compare Addis against?',
])

out_pptx = OUTPUT_DIR / 'warren_meeting_slides.pptx'
prs.save(out_pptx)
print(f'Saved deck: {out_pptx}')

## Notes for Ahmad

- Run cells top-to-bottom. Figures land in `output/warren_meeting/figures/`, deck at `output/warren_meeting/warren_meeting_slides.pptx`.
- IMPROVE data: read directly from `IMPROVE_DIR` (the Google Drive folder). Set `AETHMODULAR_IMPROVE_DIR` if your path differs from the default. Make sure the drive is mounted before running.
- Warren's R/T figure: save the digitised figure from his paper as `tmp_warren_pages/warren_RT.png` to add it as a comparison panel after slide 10.
- Ann's slide 4 says 'Fabs vs FTIR EC' which is effectively the same as slide 3 with an explicit 'FTIR' axis. If you want the duplicate to be ChemSpec EC instead, swap the column in the slide-3 cell.
- Per Ann: keep raw Fabs in 1/Mm (do NOT divide by MAC = 10) so slope = MAC; this is already the case in every plot here.
- Ann emphasised making fonts and dot sizes large. The default rcParams in this notebook target font 14, scatter `s=28-36`. Bump them up if Warren needs more contrast on screen.